<h1>Coding Session #3 - Convolutional Neural Network</h1>

Führen Sie folgende Zelle aus, um alle benötigten Bibliotheken zu installieren:

In [ ]:
!pip install -r requirements.txt

## Ziel

In dieser Coding Session soll ein Faltungsnetz (Convolutional Neural Network - CNN) implementiert und trainiert werden. CCNs sind besonders effizient für die Bild- oder Videoverarbeitung. Da es sich bei Bildern um hochdimensionale Inputs handelt, sind Fully Connected (FC) Layer ungeeignet.

## 1 Convolutional Layer

Um ein RGB-Bild mit $1,920\times 1,080$ Pixeln zu repräsentieren, bräuchte man einen FC Layer mit 6.2 Mio. Neuronen:

<img src="images/curse_of_dimensionality.png" width="600px"></img>

Hier schafft das CNN Abhilfe. Dieses nutzt Convolutional Layer. Jeder Convolutional Layer hat eine bestimmte Anzahl an Kerneln, mit denen Faltungsoperationen auf den Input angewendet werden. Dafür slidet der Kernel Spalte für Spalte und Zeile für Zeile über das Input-Bild. An jeder Stelle werden die Pixel der Region des Inputbildes $X$ elementweise mit den Gewichtungen des Kernels $K$ multipliziert und anschließend die Summe der Produkte gebildet, um den Output an der Stelle $Y[m, n]$ zu erhalten. 

<img src="images/convolution001.png" width="600px"></img>

Mit Hilfe von Faltungen lassen sich Merkmale (Features) wie Ecken oder Kanten im Bild erkennen. Die Gewichtungen der Kernels sind lernbar. Somit kann das neuronale Netz daraufhin optimiert werden, aussagekräftige Features für den Input zu extrahieren.

Folgendes Beispiel visualisiert eine Faltung mit einem Kernel zur Detektion von vertikalen Kanten:

<img src="images/conv_animation.gif" width="600px"></img>

In [ ]:
from data import CornersAndEdgesDataset
import visualization as vis


ds = CornersAndEdgesDataset(num_samples=1000, image_size=256)


labels_txt = [ds.LABELS[label] for label in ds.labels]

vis.plot_images(ds.images, labels_txt)

> <span style="color:#00A1E3">**Aufgabe 1 - Convolutional Layer**</span>
> 1. Erstellen Sie eine Klasse `ConvLayer`, welche von `Module` ableitet.
> 2. Erstellen Sie den Konstruktor (`def __init__(self, in_channels, out_channels, kernel_size)`)
>       - Der Parameter `in_channels` steht für die Anzahl der Input-Kanäle des Layers. Bei einem RGB-Bild wären dies bspw. 3 Kanäle.
>       - Der Parameter `out_channels` steht für die Anzahl der Output-Kanäle des Layers. 
>       - Initialisieren Sie die Gewichtungen der 

In [ ]:
import numpy as np
from model import Module

# Your code here


In [ ]:
import numpy as np
from model import Module

# Your code here


In [ ]:
from data import CornersAndEdgesDataset, one_hot_encode
from metrics import compute_accuracy, CrossEntropyLossWithSoftmax
from model import NeuralNetwork, ReLU, Flatten, DenseLayer
from tqdm import tqdm
import visualization as vis
import numpy as np

EPOCHS          = 20
BATCH_SIZE      = 32
LEARNING_RATE   = 0.1
IMAGE_SIZE      = 8
VIS_KERNELS     = 8

# Demo-Modus für klar sichtbare Edge/Corner-Kernels im 1. Conv-Layer
ds = CornersAndEdgesDataset(
    num_samples=50_000,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    augment=False,
    seed=42,
 )

model = NeuralNetwork(modules=[
    MaxPool(kernel_size=4, stride=4),
    ConvLayer(in_channels=1, out_channels=8, kernel_size=2),
    #ReLU(),
    
    #MaxPool(kernel_size=2, stride=2),
    Flatten(),
    #DenseLayer(in_features=8, out_features=len(ds.LABELS)),
])


model.print_shapes(ds.images[0:1])

criterion = CrossEntropyLossWithSoftmax()
dense_layers = [m for m in model.modules if isinstance(m, DenseLayer)]
conv_layers = [m for m in model.modules if hasattr(m, "kernels")]

# Feste Referenz für den visuellen Vergleich
sample_image = ds.images[0, 0].copy()

print("Vor Training: First Layer Kernels")
vis.visualize_first_layer_kernels(model, max_kernels=VIS_KERNELS)

# Training
for epoch in range(EPOCHS):
    losses_train, accs_train = [], []
    prog_train = tqdm(ds, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)

    for X, Y in prog_train:
        Y = one_hot_encode(Y, num_classes=len(ds.LABELS))

        Y_hat = model.forward(X)
        loss = criterion.forward(Y_hat, Y)

        acc = compute_accuracy(Y_hat, Y)


        losses_train.append(loss)
        accs_train.append(acc)


        grad_outputs = criterion.backward()
        model.backward(grad_outputs * LEARNING_RATE)

        prog_train.set_description(
            f"Epoch {epoch+1}/{EPOCHS}, Loss: {np.mean(losses_train):.4f}, Accuracy: {np.mean(accs_train):.4f}"
        )


    print(
        f"Epoch {epoch+1}/{EPOCHS}, Loss: {np.mean(losses_train):.4f}, "
        f"Accuracy: {np.mean(accs_train):.4f}"
    )

print("Nach Training: First Layer Kernels")
vis.visualize_first_layer_kernels(model, max_kernels=VIS_KERNELS)

print("Nach Training: Feature Maps (gleiches Preprocessing wie im Training)")
for image in ds.images:
    vis.visualize_features(image[0], model, n_kernels=VIS_KERNELS, zscore_input=False)